In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
df = pd.read_csv("Telco-Customer-Churn.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df.dropna(inplace=True)

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

df.drop("customerID", axis=1, inplace=True)

df.head()

In [ ]:
sns.countplot(x="Churn", data=df)
plt.title("Churn Distribution")
plt.show()

In [ ]:
sns.boxplot(x="Churn", y="tenure", data=df)
plt.title("Tenure vs Churn")
plt.show()

In [ ]:
sns.boxplot(x="Churn", y="MonthlyCharges", data=df)
plt.title("Monthly Charges vs Churn")
plt.show()

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True)
df_encoded.head()

In [ ]:
X = df_encoded.drop("Churn", axis=1)
y = df_encoded["Churn"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_pred))
print(confusion_matrix(y_test, lr_pred))
print(classification_report(y_test, lr_pred))

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print(confusion_matrix(y_test, rf_pred))
print(classification_report(y_test, rf_pred))

In [ ]:
rf_balanced = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)

rf_balanced.fit(X_train, y_train)

rf_pred_bal = rf_balanced.predict(X_test)

print("Balanced Random Forest Accuracy:", accuracy_score(y_test, rf_pred_bal))
print(confusion_matrix(y_test, rf_pred_bal))
print(classification_report(y_test, rf_pred_bal))

In [ ]:
y_prob = rf_balanced.predict_proba(X_test)[:, 1]

threshold = 0.30
y_pred_custom = (y_prob >= threshold).astype(int)

print("Custom Threshold Random Forest")
print(confusion_matrix(y_test, y_pred_custom))
print(classification_report(y_test, y_pred_custom))

In [ ]:
from sklearn.metrics import recall_score

print("Old RF Churn Recall:", recall_score(y_test, rf_pred))
print("Balanced RF Churn Recall:", recall_score(y_test, rf_pred_bal))
print("Custom Threshold RF Churn Recall:", recall_score(y_test, y_pred_custom))

In [ ]:
from sklearn.metrics import roc_auc_score

print("ROC–AUC Score:", roc_auc_score(y_test, y_prob))

In [ ]:
importance = pd.Series(
    rf.feature_importances_, index=X.columns
).sort_values(ascending=False)

importance.head(10)

In [ ]:
importance.head(10).plot(kind="bar", figsize=(8,5))
plt.title("Top 10 Features Influencing Churn")
plt.show()